In [2]:
import numpy as np

def generate_toy_problem(m=100, n=50):
    """
    Generates a random linear system Ax = b.
    m: number of equations (rows)
    n: number of variables (cols)
    """
    # 1. Generate random matrix A
    A = np.random.randn(m, n)
    
    # 2. Create a known solution x_true
    x_true = np.random.randn(n)
    
    # 3. Compute b so that Ax = b is guaranteed to be consistent
    b = A @ x_true
    
    return A, b, x_true

np.random.seed(20446163)  # For reproducibility
A, b, x_true = generate_toy_problem(100, 100)
print(f"Condition Number: {np.linalg.cond(A):.2f}")

Condition Number: 210.19


In [3]:
# Solve the system in its original domainusing matrix inversion and check the error
x_solved = np.linalg.solve(A, b)
error = np.linalg.norm(x_solved - x_true)
print(f"Solution Error: {error:.2e}")

# Solve the system after applying min-max normalization to A and b
M = np.hstack((A, b.reshape(-1, 1)))  # Combine A and b for normalization
M_min = M.min(axis=0)
M_max = M.max(axis=0)
M_normalized = (M - M_min) / (M_max - M_min)
A_normalized = M_normalized[:, :-1]
b_normalized = M_normalized[:, -1]
x_solved_normalized = np.linalg.solve(A_normalized, b_normalized)

Solution Error: 6.90e-14


In [4]:
# Solve the system in its original domain using the Kaczmarz method
def kaczmarz(A, b, x0=None, max_iters=10000):
    m, n = A.shape
    if x0 is None:
        x = np.zeros(n)
    else:
        x = x0.copy()
    
    for it in range(max_iters):
        for i in range(m):
            ai = A[i]
            bi = b[i]
            # Update the solution estimate
            residual = bi - ai @ x
            x += (residual / np.dot(ai, ai)) * ai
            
    return x
x_kaczmarz = kaczmarz(A, b)
error_kaczmarz = np.linalg.norm(x_kaczmarz - x_true)
print(f"Kaczmarz Solution Error: {error_kaczmarz:.2e}")

# Solve the system after applying min-max normalization using the Kaczmarz method
x_kaczmarz_normalized = kaczmarz(A_normalized, b_normalized)
error_kaczmarz_normalized = np.linalg.norm(x_kaczmarz_normalized - x_solved_normalized)
print(f"Kaczmarz Normalized Solution Error: {error_kaczmarz_normalized:.2e}")

Kaczmarz Solution Error: 1.11e-01
Kaczmarz Normalized Solution Error: 1.58e-01
